# Step 0.5 - ChemPerturb-Bridge latent perturbation embeddings

**What this notebook does.** For every JUMP compound it computes a 256-number embedding from the
chemical structure, using a *compound encoder* intended to place molecules in a "latent perturbation
space" - a space meant to reflect the gene-expression response a compound causes, rather than its
structure alone.

**How.** SMILES → [RDKit](https://www.rdkit.org) molecule → Morgan fingerprint (a 2,048-bit
description of the molecule's local substructures, radius 2) → encoder network
(2,048 → 512 → 256 numbers) → embedding. The code lives in `src/model_benchmarking/chemperturb_bridge.py`,
which can also be run from the command line:
`uv run python -m model_benchmarking.chemperturb_bridge --checkpoint /path/to/weights.ckpt`.

> ⚠️ **Important - the saved embeddings come from an untrained network.** The encoder only produces
> meaningful embeddings when trained weights are loaded (`CHECKPOINT_PATH` below). The saved run printed
> *"Checkpoint path not provided or not found. Initializing base LPM compound encoder."* (Step 3), i.e.
> the network weights were **random**. The resulting `jump_cp_latent_perturbation_embeddings.parquet` is
> therefore a random projection of Morgan fingerprints: it encodes chemical structure, not learned
> perturbation biology, and should not be used as ChemPerturb-Bridge output until it is regenerated with
> trained weights.
>
> Also note that the checkpoint loader calls `load_state_dict(..., strict=False)`, which silently skips
> any weights whose names do not match. When a checkpoint is supplied, confirm that every layer was
> actually loaded. None of the analysis notebooks in this repository use this file.

**Input.** JUMP compound metadata with standardized SMILES (`JUMP_CP_compound_metadata_with_SMILES_standard.csv`).

**Output.** `ChemPerturb-Bridge/latent_perturbation_embeddings/jump_cp_latent_perturbation_embeddings.parquet`
(115,689 of 115,796 compounds could be featurized).

**Where to run it.** The saved run used Google Colab: the original notebook built a separate Python 3.10
environment, wrote the script to disk with `%%writefile` and ran it. The script is now part of the
`model_benchmarking` package, so the pipeline runs directly in the project environment (`uv sync`).
Raw downloads are cached under the repository-level `data/raw/chemperturb_bridge/` directory; the generated Parquet table is retained under this stage's `results/` directory and uploaded to GCS.
The outputs below are those of the original run.

## Setup

Locally, `uv sync` installs everything needed; on Google Colab, first run
`!pip install git+https://github.com/axiomcura/ModelBenchmarking.git`.

In [1]:
import numpy as np

from model_benchmarking.chemperturb_bridge import (
    DEVICE,
    INPUT_GCS_URI,
    LATENT_DIM,
    OUTPUT_GCS_URI,
    compute_latent_embeddings,
    featurize_compounds,
    find_smiles_column,
    get_encoder_model,
    load_compound_metadata,
    save_and_upload,
)
from model_benchmarking.paths import notebook_artifact_dirs


## Configuration

`CHECKPOINT_PATH` points to trained encoder weights; `None` reproduces the saved run (random weights - see the
warning above). Raw source files and generated results use the standard project artifact directories.

In [2]:
CHECKPOINT_PATH = None
RAW_DATA_DIR, RESULTS_DIR = notebook_artifact_dirs("0.download_and_assemble", "chemperturb_bridge")

## Extract the embeddings

### Step 1 - Download and load the compound metadata
One row per JUMP compound, with its standardized SMILES.

In [3]:
df = load_compound_metadata(INPUT_GCS_URI, str(RAW_DATA_DIR))

### Step 2 - Featurize the structures
Each SMILES is parsed with RDKit into its canonical SMILES, InChIKey and Morgan fingerprint. Compounds whose SMILES
cannot be parsed keep an empty fingerprint and later get empty (NaN) embeddings.

In [4]:
smiles_col = find_smiles_column(df)
fp_list = featurize_compounds(df, smiles_col)

### Step 3 - Encode the fingerprints
Fingerprints pass through the compound encoder in batches of 512, in evaluation mode and without gradients.

In [5]:
print(f"🧠 Computing latent embeddings on device: {DEVICE}...")
model = get_encoder_model(CHECKPOINT_PATH, latent_dim=LATENT_DIM, device=DEVICE)
latent = compute_latent_embeddings(fp_list, model)

### Step 4 - Save and upload
The metadata plus 256 embedding columns (`latent_pert_dim_0 … latent_pert_dim_255`) are written to Parquet and
uploaded to Cloud Storage.

In [6]:
result_df = save_and_upload(df, latent, OUTPUT_GCS_URI, str(RESULTS_DIR))

### Summary

In [7]:
valid_count = int(np.isfinite(latent[:, 0]).sum())
print("=" * 70)
print("✅ PIPELINE COMPLETED SUCCESSFULLY!")
print(f"   Destination URI: {OUTPUT_GCS_URI}")
print(f"   Total rows with embeddings: {valid_count:,}")
print("=" * 70)